In [13]:
import requests 
from bs4 import BeautifulSoup
from io import StringIO
from datetime import datetime
import pandas as pd

In [2]:
def log_progress(message):

    with open ("./input/information.txt","a") as f:
        f.write(f"datatime : {datetime.now()} | {message}\n")

In [26]:
def extract(url,table_attri):

    response = requests.get(url)
    soup = BeautifulSoup(response.text,'html.parser')

    table = soup.find('span',string =table_attri).find_next('table')

    df = pd.read_html(StringIO(str(table)))[0]

    log_progress("EXTRACTION OF DATA HAS BEEN COMPLETED.")

    return df

In [ ]:
def transform(df,csv_path):

    exchange_currencies = pd.read_csv(csv_path,index_col= 0).to_dict()['Rate']
    
    df['MC_GBP_Billion'] = round(df['Market cap (US$ billion)'] * exchange_rate['GBP'], 2)
    df['MC_EUR_Billion'] = round(df['Market cap (US$ billion)'] * exchange_rate['EUR'], 2)
    df['MC_INR_Billion'] = round(df['Market cap (US$ billion)'] * exchange_rate['INR'], 2)
    log_progress("TRANSFORMATION OF DATA IS SUCCESSFUL.")
    return df

In [ ]:
def load(df,output_path):
    df.to_csv(output_path)

    log_progress("LOADING TO CSV FILE SUCCESSFULLY COMPLETED.")
    return df

In [ ]:
if __name__ == '__main__':
    url = ""
    table_atrri =  ''
    csv_path = ''
    output_path = ''
    log_progress("ETL PIPELINE COMPLETED FULLY ALLUMDUILLAH.")

In [1]:

import requests
import pandas as pd

headers = {
    "User-Agent": "Mozilla 5.0 (Windows NT 10.0; Win64; x64) AppleWebKit 537.36 (KHTML, like Gecko) Chrome 115.0.0.0 Safari 537.36"
}
url = requests.get('https://www.pakwheels.com/used-cars/karachi/24857.json?client_id=37952d7752aae22726aff51be531cddd&client_secret=014a5bc91e1c0f3af4ea6dfaa7eee413&api_version=19&extra_info=true', headers=headers)
data = url.json()
cars = data.get('result', [])
df = pd.DataFrame(cars)
print(df.head())

ConnectionError: HTTPSConnectionPool(host='www.pakwheels.com', port=443): Max retries exceeded with url: /used-cars/karachi/24857.json?client_id=37952d7752aae22726aff51be531cddd&client_secret=014a5bc91e1c0f3af4ea6dfaa7eee413&api_version=19&extra_info=true (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001A0A80AA7B0>: Failed to resolve 'www.pakwheels.com' ([Errno 11001] getaddrinfo failed)"))

In [12]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
import json
headers = {
    "User-Agent": "Mozilla 5.0 (Windows NT 10.0; Win64; x64) AppleWebKit 537.36 (KHTML, like Gecko) Chrome 115.0.0.0 Safari 537.36"
}
url = requests.get('https://www.pakwheels.com/used-cars/karachi/24857.json?client_id=37952d7752aae22726aff51be531cddd&client_secret=014a5bc91e1c0f3af4ea6dfaa7eee413&api_version=19&extra_info=true', headers=headers)
soup = BeautifulSoup(url.text,'html.parser')
data = url.json()
cars = data.get(StringIO(str('result')), [])

df = pd.DataFrame(cars)
print(df.head())

Empty DataFrame
Columns: []
Index: []


In [13]:
import requests
import pandas as pd
import json

# ================== EXTRACT ==================
def extract(url, headers):
    """
    Extract data from PakWheels API
    
    Args:
        url: API endpoint URL
        headers: Request headers
        
    Returns:
        dict: Raw JSON data from API
    """
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise error for bad status codes
        
        # Since it's a JSON API, parse directly
        data = response.json()
        print(f"✓ Data extracted successfully. Found {len(data.get('results', []))} cars.")
        return data
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Error extracting data: {e}")
        return None


# ================== TRANSFORM ==================
def transform(data):
    """
    Transform raw JSON data into a cleaned pandas DataFrame
    
    Args:
        data: Raw JSON data from API
        
    Returns:
        pd.DataFrame: Transformed dataframe
    """
    try:
        # Extract the 'results' key (not 'result')
        cars = data.get('results', [])
        
        if not cars:
            print("✗ No cars data found in the response")
            return pd.DataFrame()
        
        # Convert to DataFrame
        df = pd.DataFrame(cars)
        
        # Display basic info
        print(f"✓ Data transformed. Shape: {df.shape}")
        print(f"✓ Columns: {list(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"✗ Error transforming data: {e}")
        return pd.DataFrame()


# ================== LOAD ==================
def load(df, filename='pakwheels_karachi_cars.csv'):
    """
    Load DataFrame to CSV file
    
    Args:
        df: pandas DataFrame
        filename: Output CSV filename
        
    Returns:
        bool: Success status
    """
    try:
        if df.empty:
            print("✗ DataFrame is empty. Nothing to save.")
            return False
        
        # Save to CSV
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✓ Data loaded successfully to '{filename}'")
        print(f"✓ Total records saved: {len(df)}")
        return True
        
    except Exception as e:
        print(f"✗ Error loading data: {e}")
        return False


# ================== MAIN ETL PIPELINE ==================
def main():
    """
    Main ETL pipeline for PakWheels data
    """
    print("="*50)
    print("PakWheels Karachi Used Cars - ETL Pipeline")
    print("="*50)
    
    # Configuration
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    
    url = "https://www.pakwheels.com/used-cars/karachi/24857"
    
    # ETL Process
    print("\n[1/3] EXTRACT - Fetching data from API...")
    raw_data = extract(url, headers)
    
    if raw_data:
        print("\n[2/3] TRANSFORM - Converting to DataFrame...")
        df = transform(raw_data)
        
        if not df.empty:
            # Display sample data
            print("\nSample Data (First 5 rows):")
            print(df.head())
            
            print("\n[3/3] LOAD - Saving to CSV file...")
            load(df, 'pakwheels_karachi_cars.csv')
        else:
            print("\n✗ Transform failed. Skipping load step.")
    else:
        print("\n✗ Extract failed. Pipeline terminated.")
    
    print("\n" + "="*50)
    print("ETL Pipeline Complete!")
    print("="*50)


# Run the pipeline
if __name__ == "__main__":
    main()

PakWheels Karachi Used Cars - ETL Pipeline

[1/3] EXTRACT - Fetching data from API...
✗ Error extracting data: Expecting value: line 1 column 1 (char 0)

✗ Extract failed. Pipeline terminated.

ETL Pipeline Complete!


In [15]:
import requests
import pandas as pd
import json

# ================== EXTRACT ==================
def extract(url, headers):
    """
    Extract data from PakWheels API
    
    Args:
        url: API endpoint URL
        headers: Request headers
        
    Returns:
        dict: Raw JSON data from API
    """
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        response.raise_for_status()  # Raise error for bad status codes
        
        # Try to parse as JSON
        data = response.json()
        print(f"✓ Data extracted successfully. Found {len(data.get('result', []))} cars.")
        return data
        
    except json.JSONDecodeError as e:
        print(f"✗ JSON Decode Error: {e}")
        print(f"✗ Response is not valid JSON. The API might require authentication or returned HTML.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"✗ Request Error: {e}")
        return None


# ================== TRANSFORM ==================
def transform(data):
    """
    Transform raw JSON data into a cleaned pandas DataFrame
    
    Args:
        data: Raw JSON data from API
        
    Returns:
        pd.DataFrame: Transformed dataframe
    """
    try:
        # Extract the 'result' key (singular, not plural)
        cars = data.get('result', [])
        
        if not cars:
            print("✗ No cars data found in the response")
            return pd.DataFrame()
        
        # Convert to DataFrame
        df = pd.DataFrame(cars)
        
        # Display basic info
        print(f"✓ Data transformed. Shape: {df.shape}")
        print(f"✓ Columns: {list(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"✗ Error transforming data: {e}")
        return pd.DataFrame()


# ================== LOAD ==================
def load(df, filename='pakwheels_karachi_cars.csv'):
    """
    Load DataFrame to CSV file
    
    Args:
        df: pandas DataFrame
        filename: Output CSV filename
        
    Returns:
        bool: Success status
    """
    try:
        if df.empty:
            print("✗ DataFrame is empty. Nothing to save.")
            return False
        
        # Save to CSV
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✓ Data loaded successfully to '{filename}'")
        print(f"✓ Total records saved: {len(df)}")
        return True
        
    except Exception as e:
        print(f"✗ Error loading data: {e}")
        return False


# ================== MAIN ETL PIPELINE ==================
def main():
    """
    Main ETL pipeline for PakWheels data
    """
    print("="*50)
    print("PakWheels Karachi Used Cars - ETL Pipeline")
    print("="*50)
    
    # Configuration
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    
    url = "https://www.pakwheels.com/used-cars/karachi/24857.json?client_id=37952d7752aae22726aff51be531cddd&client_secret=014a5bc91e1c0f3af4ea6dfaa7eee413&api_version=19&extra_info=true"
    
    # ETL Process
    print("\n[1/3] EXTRACT - Fetching data from API...")
    raw_data = extract(url, headers)
    
    if raw_data:
        print("\n[2/3] TRANSFORM - Converting to DataFrame...")
        df = transform(raw_data)
        
        if not df.empty:
            # Display sample data
            print("\nSample Data (First 5 rows):")
            print(df.head())
            
            print("\n[3/3] LOAD - Saving to CSV file...")
            load(df, 'pakwheels_karachi_cars.csv')
        else:
            print("\n✗ Transform failed. Skipping load step.")
    else:
        print("\n✗ Extract failed. Pipeline terminated.")
    
    print("\n" + "="*50)
    print("ETL Pipeline Complete!")
    print("="*50)


# Run the pipeline
if __name__ == "__main__":
    main()

PakWheels Karachi Used Cars - ETL Pipeline

[1/3] EXTRACT - Fetching data from API...
✓ Data extracted successfully. Found 25 cars.

[2/3] TRANSFORM - Converting to DataFrame...
✓ Data transformed. Shape: (25, 88)
✓ Columns: ['ad_id', 'city_name', 'city_area', 'seller_comments', 'rich_description', 'price', 'monthly_installment', 'remaining_months', 'discount', 'original_price', 'title', 'url_slug', 'pictures_count', 'pictures', 'klass', 'video_link', 'video_id', 'sell_it_for_me_ad', 'sell_it_for_me_eligible', 'status', 'bumped_count', 'is_featured', 'ad_listing_id', 'last_updated', 'user', 'featured_request', 'created_at', 'is_certified', 'is_suzuki_certified', 'is_pakwheels_certified', 'certified_slug', 'certification_rating', 'inspection_rating', 'is_certified_requested', 'dealership_id', 'dealership_verified', 'certification_enabled', 'exact_color', 'disclaimer', 'warranty_text', 'recommended_oil', 'oil_widget_name', 'oil_widget_target_url', 'oil_logo', 'inspection_price', 'allow_w

In [18]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import re

# ================== EXTRACT ==================
def extract(base_url, headers, max_pages=5):
    """
    Extract car data from PakWheels HTML pages using BeautifulSoup
    
    Args:
        base_url: PakWheels search page URL
        headers: Request headers
        max_pages: Maximum number of pages to scrape
        
    Returns:
        list: List of car dictionaries
    """
    all_cars = []
    
    try:
        for page in range(1, max_pages + 1):
            print(f"\n📄 Scraping page {page}...")
            
            # Construct URL for each page
            if page == 1:
                url = base_url
            else:
                url = f"{base_url}?page={page}"
            
            # Send request
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            
            # Parse HTML with BeautifulSoup
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find all car listings - they are in <li> tags with class "classified-listing"
            car_listings = soup.find_all('li', class_='classified-listing')
            
            if not car_listings:
                print(f"⚠ No more cars found on page {page}. Stopping.")
                break
            
            print(f"✓ Found {len(car_listings)} cars on page {page}")
            
            # Extract data from each car listing
            for car in car_listings:
                car_data = extract_car_details(car)
                if car_data:
                    all_cars.append(car_data)
            
            # Be polite - add delay between requests
            time.sleep(2)
        
        print(f"\n✓ Total cars extracted: {len(all_cars)}")
        return all_cars
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Error during extraction: {e}")
        return all_cars


def extract_car_details(car_element):
    """
    Extract details from a single car listing element
    
    Args:
        car_element: BeautifulSoup element containing car info
        
    Returns:
        dict: Car details
    """
    try:
        car_data = {}
        
        # Extract car title/name
        title_elem = car_element.find('h3', class_='fs18')
        if not title_elem:
            title_elem = car_element.find('a', class_='car-name')
        car_data['title'] = title_elem.get_text(strip=True) if title_elem else 'N/A'
        
        # Extract price
        price_elem = car_element.find('div', class_='price-details')
        if not price_elem:
            price_elem = car_element.find('strong', string=re.compile(r'PKR'))
        car_data['price'] = price_elem.get_text(strip=True) if price_elem else 'N/A'
        
        # Extract location
        location_elem = car_element.find('ul', class_='list-unstyled')
        if location_elem:
            location_li = location_elem.find('li', class_='')
            if location_li:
                car_data['location'] = location_li.get_text(strip=True)
            else:
                car_data['location'] = 'N/A'
        else:
            car_data['location'] = 'N/A'
        
        # Extract specifications (year, mileage, engine, etc.)
        specs = car_element.find_all('li')
        year = 'N/A'
        mileage = 'N/A'
        engine = 'N/A'
        
        for spec in specs:
            spec_text = spec.get_text(strip=True)
            
            # Check for year (4 digit number)
            if re.search(r'\b(19|20)\d{2}\b', spec_text):
                year_match = re.search(r'\b(19|20)\d{2}\b', spec_text)
                if year_match:
                    year = year_match.group()
            
            # Check for mileage (contains 'km')
            if 'km' in spec_text.lower():
                mileage = spec_text
            
            # Check for engine info (contains 'cc' or engine type)
            if 'cc' in spec_text.lower() or 'petrol' in spec_text.lower() or 'diesel' in spec_text.lower():
                engine = spec_text
        
        car_data['year'] = year
        car_data['mileage'] = mileage
        car_data['engine'] = engine
        
        # Extract car link
        link_elem = car_element.find('a', href=True)
        if link_elem and link_elem.get('href'):
            href = link_elem['href']
            car_data['link'] = 'https://www.pakwheels.com' + href if href.startswith('/') else href
        else:
            car_data['link'] = 'N/A'
        
        # Extract posted date
        date_elem = car_element.find('div', class_='date')
        if not date_elem:
            date_elem = car_element.find('span', string=re.compile(r'(days?|hours?|weeks?)'))
        car_data['posted_date'] = date_elem.get_text(strip=True) if date_elem else 'N/A'
        
        # Check if featured
        featured = car_element.find('div', class_='featured-ribbon')
        car_data['featured'] = 'Yes' if featured else 'No'
        
        return car_data
        
    except Exception as e:
        print(f"⚠ Error extracting car details: {e}")
        return None


# ================== TRANSFORM ==================
def transform(cars_list):
    """
    Transform raw car data list into a cleaned pandas DataFrame
    
    Args:
        cars_list: List of car dictionaries
        
    Returns:
        pd.DataFrame: Transformed dataframe
    """
    try:
        if not cars_list:
            print("✗ No cars data to transform")
            return pd.DataFrame()
        
        # Convert to DataFrame
        df = pd.DataFrame(cars_list)
        
        # Data cleaning
        # Remove 'PKR' and commas from price, convert to numeric
        if 'price' in df.columns:
            df['price_clean'] = df['price'].str.replace('PKR', '', regex=False)
            df['price_clean'] = df['price_clean'].str.replace(',', '', regex=False)
            df['price_clean'] = df['price_clean'].str.strip()
            df['price_numeric'] = pd.to_numeric(df['price_clean'], errors='coerce')
            df = df.drop('price_clean', axis=1)
        
        # Extract year as numeric
        if 'year' in df.columns:
            df['year_numeric'] = pd.to_numeric(df['year'], errors='coerce')
        
        # Extract mileage as numeric (remove 'km' and commas)
        if 'mileage' in df.columns:
            df['mileage_clean'] = df['mileage'].str.replace('km', '', regex=False)
            df['mileage_clean'] = df['mileage_clean'].str.replace(',', '', regex=False)
            df['mileage_clean'] = df['mileage_clean'].str.strip()
            df['mileage_numeric'] = pd.to_numeric(df['mileage_clean'], errors='coerce')
            df = df.drop('mileage_clean', axis=1)
        
        # Remove duplicates based on title and price
        df = df.drop_duplicates(subset=['title', 'price'], keep='first')
        
        print(f"✓ Data transformed. Shape: {df.shape}")
        print(f"✓ Columns: {list(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"✗ Error transforming data: {e}")
        return pd.DataFrame()


# ================== LOAD ==================
def load(df, filename='pakwheels_karachi_cars.csv'):
    """
    Load DataFrame to CSV file
    
    Args:
        df: pandas DataFrame
        filename: Output CSV filename
        
    Returns:
        bool: Success status
    """
    try:
        if df.empty:
            print("✗ DataFrame is empty. Nothing to save.")
            return False
        
        # Save to CSV
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✓ Data loaded successfully to '{filename}'")
        print(f"✓ Total records saved: {len(df)}")
        
        # Display statistics
        if 'price_numeric' in df.columns:
            valid_prices = df['price_numeric'].dropna()
            if len(valid_prices) > 0:
                print(f"\n📊 Price Statistics:")
                print(f"   Average Price: PKR {valid_prices.mean():,.0f}")
                print(f"   Min Price: PKR {valid_prices.min():,.0f}")
                print(f"   Max Price: PKR {valid_prices.max():,.0f}")
        
        return True
        
    except Exception as e:
        print(f"✗ Error loading data: {e}")
        return False


# ================== MAIN ETL PIPELINE ==================
def main():
    """
    Main ETL pipeline for PakWheels data using BeautifulSoup
    """
    print("="*60)
    print("PakWheels Karachi Used Cars - BeautifulSoup ETL Pipeline")
    print("="*60)
    
    # Configuration
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    
    # PakWheels Karachi used cars search page
    base_url = "https://www.pakwheels.com/used-cars/search/-/ct_karachi/"
    
    max_pages = 5  # Number of pages to scrape (YOU CAN CHANGE THIS)
    
    # ETL Process
    print(f"\n[1/3] EXTRACT - Scraping data using BeautifulSoup...")
    print(f"🌐 Target: {base_url}")
    print(f"📄 Pages to scrape: {max_pages}")
    
    cars_data = extract(base_url, headers, max_pages)
    
    if cars_data:
        print(f"\n[2/3] TRANSFORM - Converting to DataFrame and cleaning...")
        df = transform(cars_data)
        
        if not df.empty:
            # Display sample data
            print("\n📋 Sample Data (First 5 rows):")
            print(df[['title', 'price', 'year', 'location']].head())
            
            print(f"\n[3/3] LOAD - Saving to CSV file...")
            load(df, 'pakwheels_karachi_cars.csv')
        else:
            print("\n✗ Transform failed. Skipping load step.")
    else:
        print("\n✗ Extract failed. Pipeline terminated.")
    
    print("\n" + "="*60)
    print("ETL Pipeline Complete!")
    print("="*60)


# Run the pipeline
if __name__ == "__main__":
    main()

PakWheels Karachi Used Cars - BeautifulSoup ETL Pipeline

[1/3] EXTRACT - Scraping data using BeautifulSoup...
🌐 Target: https://www.pakwheels.com/used-cars/search/-/ct_karachi/
📄 Pages to scrape: 2

📄 Scraping page 1...
✓ Found 25 cars on page 1

📄 Scraping page 2...
✓ Found 28 cars on page 2

✓ Total cars extracted: 53

[2/3] TRANSFORM - Converting to DataFrame and cleaning...
✓ Data transformed. Shape: (50, 12)
✓ Columns: ['title', 'price', 'location', 'year', 'mileage', 'engine', 'link', 'posted_date', 'featured', 'price_numeric', 'year_numeric', 'mileage_numeric']

📋 Sample Data (First 5 rows):
                                 title          price  year location
0        Toyota Raize  2020 Z for Sale   PKR 63.8lacs  2020     2020
1  Toyota Passo  2013 X for Sale6.8/10     PKR 25lacs  2013     2013
2       Daihatsu Mira  2022 L for Sale  PKR 36.25lacs  2022     2022
3        Toyota Raize  2020 Z for Sale   PKR 61.5lacs  2020     2020
4        Toyota Raize  2020 G for Sale   PKR 57.